# 1.3 Cluster HDB blocks into walkable neighbourhoods

Residential blocks are grouped into neighbourhoods of at most 300 m across, separately inside each
subzone, so that one point per cluster can stand in for all its blocks on maps and in the early
accessibility work. The method is complete-linkage agglomerative clustering with a 300 m distance
threshold, chosen over the proposal's "blocks within 200–300 m chain together" rule because chaining
merges whole towns (see `docs/clustering_decision.md` and the comparison in
`../0_Data_Exploration/clustering_experiment.ipynb`). Note that after the 2026-10-26 consultation
the walk matrix (1_07) uses individual **blocks** as origins; the clusters are kept for mapping and
the `hdb_points` layer built here is the block list the matrix is built on. Original script:
`scripts/cluster_hdb_blocks.py` (Minh).

**Inputs**
- `output/hdb_blocks.gpkg`, layer `hdb_blocks` (1_02) — all blocks with `is_residential` and unit counts.
- `output/hdb_blocks.gpkg`, layer `subzones` (1_01).

**Outputs** (all EPSG:3414, written to `output/hdb_blocks.gpkg`)
- `hdb_points` — one centroid per residential block with units (10,769): subzone, `cluster_id`,
  flat counts, `gfa_m2`, `color_idx`; origins of the walk matrix (1_07).
- `hdb_clusters` — one point per cluster (dwelling-unit-weighted centroid): `n_blocks`,
  `dwelling_units`, `gfa_m2`, `diameter_m`, `color_idx`.
- `hdb_cluster_hulls` — convex hull per cluster, for maps and visual QA only.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import pyogrio
from scipy.spatial import cKDTree
from scipy.spatial.distance import pdist
from sklearn.cluster import AgglomerativeClustering

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "bt4015").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from bt4015.paths import DATA, OUT, FIG, LARGE

GPKG = OUT / "hdb_blocks.gpkg"
DIAMETER_M = 300          # max cluster diameter, decided 2026-10-18 (docs/clustering_decision.md); 400 is the fallback
HULL_BUFFER_M = 15        # visual only, so singleton / pair clusters render as a shape
# Block attributes carried onto the point layer.
POINT_COLS = ["OBJECTID", "BLK_NO", "decoded_street", "POSTAL_COD", "year_completed",
              "max_floor_lvl", "total_dwelling_units", "footprint_m2", "gfa_m2",
              "1room_sold", "2room_sold", "3room_sold", "4room_sold", "5room_sold",
              "exec_sold", "1room_rental", "2room_rental", "3room_rental"]

## 1. Residential block centroids tagged with their subzone

Carparks, markets and pavilions are dropped (`is_residential`), as are residential blocks with no
dwelling units. A centroid lying exactly on a subzone boundary falls in no polygon with `within`,
so those few are snapped to the nearest subzone instead of being lost.

In [2]:
# 1. Residential blocks with units: the demand-side block list (10,769).
blocks = pyogrio.read_dataframe(GPKG, layer="hdb_blocks")
blocks = blocks[blocks["is_residential"] & blocks["total_dwelling_units"].gt(0)].copy()
pts = blocks[POINT_COLS + ["geometry"]].copy()
pts["geometry"] = blocks.geometry.centroid

# 2. Tag each centroid with its subzone; clustering runs within subzones.
sz = pyogrio.read_dataframe(GPKG, layer="subzones")[["SUBZONE_C", "SUBZONE_N", "PLN_AREA_N", "geometry"]]
pts = gpd.sjoin(pts, sz, how="left", predicate="within").drop(columns="index_right")

# 3. Centroids on a boundary match no polygon: snap them to the nearest subzone.
miss = pts["SUBZONE_C"].isna()
if miss.any():
    near = gpd.sjoin_nearest(pts.loc[miss, ["geometry"]], sz, how="left")
    pts.loc[miss, ["SUBZONE_C", "SUBZONE_N", "PLN_AREA_N"]] = near[["SUBZONE_C", "SUBZONE_N", "PLN_AREA_N"]].values
print(f"{len(pts):,} residential block centroids tagged to {pts['SUBZONE_C'].nunique()} subzones "
      f"({int(miss.sum())} snapped to nearest subzone)")

10,769 residential block centroids tagged to 159 subzones (0 snapped to nearest subzone)


## 2. Complete-linkage clustering, 300 m, within each subzone

Complete linkage only merges two groups if *every* pair of blocks across them is within the
threshold, so the threshold is a hard cap on cluster diameter. Running it per subzone keeps every
cluster inside one subzone, which keeps the population allocation clean. A subzone with a single
block is its own cluster (sklearn needs at least two samples).

In [3]:
# 1. Coordinates in metres; one AgglomerativeClustering fit per subzone.
xy = np.c_[pts.geometry.x, pts.geometry.y]
cid = np.empty(len(pts), dtype=object)
for code_, idx in pts.groupby("SUBZONE_C").indices.items():
    if len(idx) == 1:
        lab = np.zeros(1, int)
    else:
        lab = AgglomerativeClustering(n_clusters=None, distance_threshold=DIAMETER_M,
                                      linkage="complete").fit_predict(xy[idx])
    # 2. Cluster id = subzone code + running number, so ids are unique nationwide and readable.
    cid[idx] = [f"{code_}_{l:03d}" for l in lab]
pts["cluster_id"] = cid
print(f"{pd.Series(cid).nunique():,} clusters across {pts['SUBZONE_C'].nunique()} subzones")

1,403 clusters across 159 subzones


## 3. Cluster summary: weighted centroid, size, diameter, map colour

The cluster point is the dwelling-unit-weighted centroid, so it sits where the people are.
`diameter_m` is the largest block-to-block distance and is asserted to respect the cap.
`color_idx` (0–7) is a greedy graph colouring over each cluster's eight nearest neighbours so that
adjacent clusters never share a colour: in QGIS use Categorized symbology on `color_idx`.

In [4]:
def colour_clusters(centroids_xy, k_neighbours=8, n_colours=8):
    """Greedy colouring: each cluster takes the lowest colour not used by its k nearest neighbours."""
    tree = cKDTree(centroids_xy)
    nb = tree.query(centroids_xy, k=min(k_neighbours + 1, len(centroids_xy)))[1]
    col = np.full(len(centroids_xy), -1)
    for i in range(len(centroids_xy)):
        used = {col[j] for j in nb[i][1:] if col[j] >= 0}
        col[i] = next((c for c in range(n_colours) if c not in used), i % n_colours)
    return col


# 1. Per-cluster aggregates from a plain DataFrame (faster than a GeoDataFrame groupby).
w = pts["total_dwelling_units"].to_numpy(float)
df = pd.DataFrame({"cluster_id": cid, "x": xy[:, 0], "y": xy[:, 1], "w": w,
                   "gfa": pts["gfa_m2"].to_numpy(), "SUBZONE_C": pts["SUBZONE_C"].values,
                   "SUBZONE_N": pts["SUBZONE_N"].values, "PLN_AREA_N": pts["PLN_AREA_N"].values})
g = df.groupby("cluster_id")
cl = pd.DataFrame({
    "SUBZONE_C": g["SUBZONE_C"].first(), "SUBZONE_N": g["SUBZONE_N"].first(),
    "PLN_AREA_N": g["PLN_AREA_N"].first(),
    "n_blocks": g.size(), "dwelling_units": g["w"].sum().astype(int), "gfa_m2": g["gfa"].sum(),
    "cx": g.apply(lambda d: np.average(d["x"], weights=d["w"]), include_groups=False),
    "cy": g.apply(lambda d: np.average(d["y"], weights=d["w"]), include_groups=False),
    "diameter_m": g.apply(lambda d: pdist(d[["x", "y"]].to_numpy()).max() if len(d) > 1 else 0.0,
                          include_groups=False),
}).reset_index()

# 2. The cap must hold for every cluster; complete linkage guarantees it.
assert cl["diameter_m"].max() <= DIAMETER_M + 1e-6

# 3. Map colours, copied back onto the points through the cluster id.
cl["color_idx"] = colour_clusters(cl[["cx", "cy"]].to_numpy())
pts["color_idx"] = pts["cluster_id"].map(cl.set_index("cluster_id")["color_idx"])

s = cl["n_blocks"]
print(f"{len(cl):,} clusters (max diameter {DIAMETER_M} m): blocks per cluster median {s.median():.0f}, "
      f"max {s.max()}, singletons {(s == 1).sum()}; median {cl['dwelling_units'].median():.0f} flats per cluster; "
      f"95th-pct diameter {cl['diameter_m'].quantile(.95):.0f} m")
print(cl["color_idx"].value_counts().sort_index().rename("clusters per colour").to_string())

1,403 clusters (max diameter 300 m): blocks per cluster median 7, max 31, singletons 6; median 788 flats per cluster; 95th-pct diameter 289 m
color_idx
0    257
1    244
2    223
3    213
4    204
5    160
6     78
7     24


## 4. Cluster points and hulls as GeoDataFrames

In [5]:
# 1. Cluster points at the weighted centroid.
clusters = gpd.GeoDataFrame(cl.drop(columns=["cx", "cy"]),
                            geometry=gpd.points_from_xy(cl["cx"], cl["cy"]), crs=pts.crs)

# 2. Convex hull per cluster, buffered so one- and two-block clusters still show as a shape.
hulls = pts.dissolve(by="cluster_id")[["geometry"]].convex_hull.buffer(HULL_BUFFER_M)
hulls = gpd.GeoDataFrame(cl.set_index("cluster_id").drop(columns=["cx", "cy"]),
                         geometry=hulls, crs=pts.crs).reset_index()
print(f"clusters: {len(clusters):,} points; hulls: {len(hulls):,} polygons, median area {hulls.area.median() / 1e4:.1f} ha")

clusters: 1,403 points; hulls: 1,403 polygons, median area 2.3 ha


## 5. Write the three layers

In [6]:
pts.to_file(GPKG, layer="hdb_points", driver="GPKG")
clusters.to_file(GPKG, layer="hdb_clusters", driver="GPKG")
hulls.to_file(GPKG, layer="hdb_cluster_hulls", driver="GPKG")
for layer in ["hdb_points", "hdb_clusters", "hdb_cluster_hulls"]:
    info = pyogrio.read_info(GPKG, layer=layer)
    print(f"{GPKG.relative_to(ROOT)} layer {layer}: {info['features']:,} features, {info['crs']}")

output/hdb_blocks.gpkg layer hdb_points: 10,769 features, EPSG:3414
output/hdb_blocks.gpkg layer hdb_clusters: 1,403 features, EPSG:3414
output/hdb_blocks.gpkg layer hdb_cluster_hulls: 1,403 features, EPSG:3414
